# SolarROI AI — Renewable Energy Investment Feasibility Assistant

**SDG Alignment:** SDG 7 (Affordable and Clean Energy), SDG 12 (Responsible Consumption and Production)

This notebook demonstrates an AI-powered pipeline that helps homeowners and MSMEs evaluate 
the financial feasibility of rooftop solar investments. It combines:

- A **financial calculator** (payback period, IRR)
- A **Retrieval-Augmented Generation (RAG)** system over official government subsidy policy 
  documents (MNRE, PM Surya Ghar, Gujarat SURYA scheme)
- **IBM Granite** (running locally) for grounded question-answering and report generation
- An **agentic pipeline** that chains retrieval, calculation, and generation into one automated 
  feasibility report

**AI components used:** Prompt engineering, IBM Granite Models, Retrieval-Augmented Generation, 
Agentic AI workflows, Entity extraction & summarization, IBM Bob (for development).

In [ ]:
!pip install transformers accelerate torch


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from getpass import getpass
hf_token = getpass("Paste your Hugging Face token: ")

In [3]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="ibm-granite/granite-4.0-h-350m",
    resume_download=True,
    token=hf_token
)

c:\Users\ACER\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\utils\_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

'C:\\Users\\ACER\\.cache\\huggingface\\hub\\models--ibm-granite--granite-4.0-h-350m\\snapshots\\3b17b717b8f2f5d305b0a92c1491e239aeda19c8'

In [15]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "ibm-granite/granite-4.0-h-350m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/370 [00:00<?, ?it/s]

In [5]:
chat = [{"role": "user", "content": "In one sentence, what is a payback period in project finance?"}]

# Step 1: turn the chat into a formatted text string (not tokens yet)
prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

# Step 2: tokenize that text properly
inputs = tokenizer(prompt, return_tensors="pt")

# Step 3: generate
output = model.generate(**inputs, max_new_tokens=100)

print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `mamba_chunk_scan_combined` is falling back to its reference PyTorch implementation because `mamba_ssm` is not installed. This is correct but much slower; install `mamba_ssm` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `selective_state_update` is falling back to its reference PyTorch implementation because `mamba_ssm` is not installed. This is correct but much slower; install `mamba_ssm` for the optimized kernel.


systemYou are a helpful assistant. Please ensure responses are professional, accurate, and safe.
userIn one sentence, what is a payback period in project finance?
assistantA payback period in project finance is the time required to recover the initial investment in a project, typically calculated by dividing the total capital expenditure by the annual cash inflow.


In [ ]:
# STEP 1
!pip install numpy_financial


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 1: Financial Feasibility Calculator

Computes the payback period and Internal Rate of Return (IRR) for a proposed solar 
installation, based on system cost, applicable government subsidy, and estimated monthly 
electricity savings.

In [21]:
import numpy_financial as npf

def calculate_solar_feasibility(system_cost, subsidy_percent, monthly_savings, system_lifetime_years=25):
    """
    Calculates payback period and IRR for a solar installation.

    Parameters:
    - system_cost: total upfront cost of the solar system (INR)
    - subsidy_percent: government subsidy as a percentage (e.g. 30 for 30%)
    - monthly_savings: estimated monthly electricity bill savings (INR)
    - system_lifetime_years: expected lifespan of the system (default 25 years)

    Returns:
    A dictionary with net cost, payback period (years), and IRR (%)
    """
    # Step 1: Apply subsidy to get the actual amount the person pays
    subsidy_amount = system_cost * (subsidy_percent / 100)
    net_cost = system_cost - subsidy_amount

    # Step 2: Convert monthly savings into annual savings
    annual_savings = monthly_savings * 12

    # Step 3: Payback period = how many years to recover net_cost
    if annual_savings > 0:
        payback_period = net_cost / annual_savings
    else:
        payback_period = float('inf')  # never pays back if no savings

    # Step 4: Build a cash flow list for IRR calculation
    # Year 0: you pay net_cost (negative cash flow)
    # Every year after: you receive annual_savings (positive cash flow)
    cash_flows = [-net_cost] + [annual_savings] * system_lifetime_years

    try:
        irr = npf.irr(cash_flows) * 100  # convert to percentage
    except Exception:
        irr = None  # IRR can't be calculated in some edge cases

    return {
        "net_cost_after_subsidy": round(net_cost, 2),
        "payback_period_years": round(payback_period, 2),
        "irr_percent": round(irr, 2) if irr is not None else "N/A"
    }

In [9]:
result = calculate_solar_feasibility(
    system_cost=300000,      # ₹3 lakh system
    subsidy_percent=30,      # 30% government subsidy
    monthly_savings=2500     # ₹2,500 saved per month on electricity
)

print(result)

{'net_cost_after_subsidy': 210000.0, 'payback_period_years': 7.0, 'irr_percent': 13.71}


In [ ]:
# STEP 2
!pip install pdfplumber

   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   --------- ------------------------------ 1.6/6.6 MB 7.7 MB/s eta 0:00:01
   --------------- ------------------------ 2.6/6.6 MB 6.4 MB/s eta 0:00:01
   ------------------------------ --------- 5.0/6.6 MB 7.7 MB/s eta 0:00:01
   ---------------------------------------- 6.6/6.6 MB 7.9 MB/s  0:00:00
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ------------------------ --------------- 2.4/3.8 MB 11.5 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 9.7 MB/s  0:00:00
   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   ------------------ --------------------- 1.8/3.9 MB 8.4 MB/s eta 0:00:01
   ----------------------------------- ---- 3.4/3.9 MB 8.3 MB/s eta 0:00:01
   ---------------------------------------- 3.9/3.9 MB 7.8 MB/s  0:00:00

   ---------------------------------------- 0/5 [pypdfium2]
   ---------------------------------------- 0/5 [pypdfiu


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Loading Subsidy Policy Documents

Three official solar subsidy policy documents are loaded and their text extracted:
- MNRE central government guidelines (PM Surya Ghar: Muft Bijli Yojana)
- PM Surya Ghar information booklet
- Gujarat SURYA rooftop solar scheme presentation

This raw policy text forms the knowledge base the AI assistant will search through.

In [4]:
import pdfplumber
import os

def extract_text_from_pdf(pdf_path):
    full_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                full_text += page_text + "\n"
    return full_text

pdf_folder = "data/subsidy_docs"
documents = {}

for filename in os.listdir(pdf_folder):
    if filename.endswith(".pdf"):
        path = os.path.join(pdf_folder, filename)
        print(f"Extracting: {filename}")
        documents[filename] = extract_text_from_pdf(path)

print(f"\nDone. Extracted {len(documents)} documents.")

Extracting: 1st_Aug_2024_PM_Surya_Ghar_Muft_Bijli_Yojana_Information.pdf
Extracting: Governement Guidelines.pdf
Extracting: SURYA-Gujarat Surya Urja Rooftop Yojana.pdf

Done. Extracted 3 documents.


In [5]:
for filename, text in documents.items():
    print(f"--- {filename} ---")
    print(text[:500])
    print("...\n")

--- 1st_Aug_2024_PM_Surya_Ghar_Muft_Bijli_Yojana_Information.pdf ---
PM Surya Ghar
PM Surya Ghar
Muft Bijli Yojana
Muft Bijli Yojana
Information Booklet
Information Booklet
(cid:31)(cid:30)(cid:29)(cid:28)(cid:27)(cid:29)(cid:28)(cid:26)
1. Brief about PM Surya Ghar: Muft Bijli Yojana ...............................................................................3
2. Subsidy Details ...............................................................................................................................3
3. Uttar Pradesh DISCOM Targets.......................
...

--- Governement Guidelines.pdf ---
Guidelines for PM-Surya Ghar: Muft Bijli Yojana
Central Financial Assistance to Residential Consumers
1) Background
a) The Government of India has approved the PM Surya Ghar: Muft Bijli Yojana on 29th February,
2024 to increase the share of solar rooftop capacity and empower residential households to
generate their own electricity. The scheme has an outlay of Rs 75,021 crore and is to be

In [ ]:
# STEP 3
!pip install sentence-transformers faiss-cpu

   ---------------------------------------- 0.0/739.8 kB ? eta -:--:--
   ---------------------------------------- 739.8/739.8 kB 6.2 MB/s  0:00:00
   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
   ---- ----------------------------------- 1.8/16.3 MB 9.1 MB/s eta 0:00:02
   --------- ------------------------------ 3.7/16.3 MB 8.9 MB/s eta 0:00:02
   ------------- -------------------------- 5.5/16.3 MB 8.8 MB/s eta 0:00:02
   ------------------ --------------------- 7.3/16.3 MB 8.5 MB/s eta 0:00:02
   -------------------- ------------------- 8.4/16.3 MB 7.8 MB/s eta 0:00:02
   ----------------------- ---------------- 9.4/16.3 MB 7.3 MB/s eta 0:00:01
   ------------------------- -------------- 10.2/16.3 MB 6.8 MB/s eta 0:00:01
   --------------------------- ------------ 11.0/16.3 MB 6.5 MB/s eta 0:00:01
   ----------------------------- ---------- 11.8/16.3 MB 6.2 MB/s eta 0:00:01
   ------------------------------ --------- 12.6/16.3 MB 6.0 MB/s eta 0:00:01
   -----


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 3: Building the RAG Index

The extracted document text is split into overlapping chunks and converted into numeric 
vector embeddings using a sentence-transformer model. These embeddings are stored in a 
FAISS index, enabling fast semantic search — finding the *meaning* of a query rather than 
just matching keywords.

In [7]:
def chunk_text(text, chunk_size=600, overlap=100):
    """
    Splits text into overlapping chunks so related sentences aren't cut apart awkwardly.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# Build a list of all chunks, keeping track of which document each came from
all_chunks = []
chunk_sources = []

for filename, text in documents.items():
    doc_chunks = chunk_text(text)
    all_chunks.extend(doc_chunks)
    chunk_sources.extend([filename] * len(doc_chunks))

print(f"Total chunks created: {len(all_chunks)}")
print(f"Example chunk:\n{all_chunks[0]}")

Total chunks created: 306
Example chunk:
PM Surya Ghar
PM Surya Ghar
Muft Bijli Yojana
Muft Bijli Yojana
Information Booklet
Information Booklet
(cid:31)(cid:30)(cid:29)(cid:28)(cid:27)(cid:29)(cid:28)(cid:26)
1. Brief about PM Surya Ghar: Muft Bijli Yojana ...............................................................................3
2. Subsidy Details ...............................................................................................................................3
3. Uttar Pradesh DISCOM Targets......................................................................................................4
4. District Targets


In [8]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load a small, fast embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Convert all chunks into numeric vectors
chunk_embeddings = embedder.encode(all_chunks)

# Build a FAISS index for fast similarity search
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(chunk_embeddings))

print(f"Index built with {index.ntotal} chunks.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\ACER\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ACER\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Index built with 306 chunks.


## Step 4: Semantic Retrieval

Given a natural-language question, this function embeds the query and searches the FAISS 
index for the most semantically relevant chunks across all three policy documents.

In [ ]:
# STEP 4
def retrieve_relevant_chunks(question, top_k=3):
    # Guard: empty question
    if not question or not question.strip():
        return []

    # Guard: top_k larger than index
    effective_k = min(top_k, index.ntotal)
    if effective_k == 0:
        return []

    question_embedding = embedder.encode([question])
    distances, indices = index.search(np.array(question_embedding), effective_k)

    results = []
    for idx in indices[0]:
        if idx == -1:          # FAISS sentinel — shouldn't happen now, but belt-and-suspenders
            continue
        results.append({
            "text": all_chunks[idx],
            "source": chunk_sources[idx]
        })
    return results

In [12]:
question = "What is the subsidy amount for a rooftop solar system?"
results = retrieve_relevant_chunks(question)

# Also capture distances for debugging
question_embedding = embedder.encode([question])
distances, _ = index.search(np.array(question_embedding), 3)

for i, (r, dist) in enumerate(zip(results, distances[0])):
    print(f"--- Match {i+1} | source: {r['source']} | L2 distance: {dist:.4f} ---")
    print(r['text'])
    print()

--- Match 1 | source: 1st_Aug_2024_PM_Surya_Ghar_Muft_Bijli_Yojana_Information.pdf | L2 distance: 0.5862 ---
PM Jan-Samarth Portal or offline mode at Banks)
For more details, please refer detailed Operational Guidelines issued by MNRE via link
https://cdnbbsr.s3waas.gov.in/s3716e1b8c6cd17b771da77391355749f3/uploads/2024/07/202407021768035484.pdf
2. Subsidy Details
S. Central Finance Total Benefit to
Type of Residential Segment State Subsidy
No. Assistance Consumer
Rooftop Solar Plant of Capacity up to 2 KW in Rs. 30,000 per KW Rs. 15,000 per KW Rs. 45,000 to Rs.
1
Residential Household or part thereof 90,000
Rs. 18,000 for No Additional Rs. 1,08,000
Additional Capacity for Plant Capacity Ranging
2 ad

--- Match 2 | source: 1st_Aug_2024_PM_Surya_Ghar_Muft_Bijli_Yojana_Information.pdf | L2 distance: 0.6480 ---
a from the Grid Connected
Rooftop Solar Phase II scheme.
 Incentives are limited to the first additional 18,000 MW of rooftop solar capacity installed
since the start of the Phase

In [ ]:
# STEP 5
def answer_question(question, top_k=3, max_new_tokens=200):
    # Step 1: Retrieve relevant chunks
    chunks = retrieve_relevant_chunks(question, top_k=top_k)
    
    if not chunks:
        return "I could not find relevant information to answer your question."

    # Step 2: Build context string from retrieved chunks
    context = "\n\n".join(
        f"[Source: {r['source']}]\n{r['text']}" for r in chunks
    )

    # Step 3: Build the chat prompt
    chat = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant. Answer the user's question using ONLY "
                "the context provided below. If the answer is not in the context, "
                "say 'I don't have enough information to answer that.'\n\n"
                f"Context:\n{context}"
            )
        },
        {
            "role": "user",
            "content": question
        }
    ]

    # Step 4: Tokenize and generate
    prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt")
    output = model.generate(**inputs, max_new_tokens=max_new_tokens)
    
    # Step 5: Decode and strip the prompt prefix — only return the new generated text
    full_text = tokenizer.decode(output[0], skip_special_tokens=True)
    # The model echoes system+user turns before the answer; split on the last "assistant" turn
    if "assistant" in full_text.lower():
        answer = full_text.split("assistant")[-1].strip()
    else:
        answer = full_text.strip()

    return answer

In [16]:
question = "What is the subsidy amount for a rooftop solar system?"
print(answer_question(question))

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `mamba_chunk_scan_combined` is falling back to its reference PyTorch implementation because `mamba_ssm` is not installed. This is correct but much slower; install `mamba_ssm` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `selective_state_update` is falling back to its reference PyTorch implementation because `mamba_ssm` is not installed. This is correct but much slower; install `mamba_ssm` for the optimized kernel.


The subsidy amount for a rooftop solar system is Rs. 45,000 per KW.


## Step 5: Grounded Question-Answering (RAG + Granite)

This function combines retrieval with generation: relevant policy text is retrieved first, 
then passed to the IBM Granite model as context, with an explicit instruction to answer 
*only* using that retrieved information — reducing the risk of the model inventing 
unsupported claims.

In [18]:
print(answer_question("What is the total budget outlay for the PM Surya Ghar scheme?"))
print(answer_question("What is SVNIT's role in the Gujarat solar scheme?"))

The total budget outlay for the PM Surya Ghar scheme is Rs 11,814 crores.
SVNIT (Surya University of Navigation and Technology) is an institution that is participating in the Gujarat solar scheme.


In [19]:
# STEP 6 
def generate_feasibility_report(system_cost, subsidy_percent, monthly_savings, location, max_new_tokens=300):
    # Step 1: Get the financial numbers
    financials = calculate_solar_feasibility(system_cost, subsidy_percent, monthly_savings)

    # Step 2: Retrieve subsidy/eligibility context relevant to the location
    policy_question = f"What are the subsidy eligibility criteria and subsidy amount for rooftop solar in {location}?"
    policy_answer = answer_question(policy_question)

    # Step 3: Build the combined prompt
    subsidy_amount = system_cost * (subsidy_percent / 100)

    chat = [
        {
            "role": "system",
            "content": (
                "You are a solar energy advisor writing for a non-expert homeowner. "
                "Write in plain, friendly language — avoid jargon. "
                "Use only the financial figures and policy information provided."
            )
        },
        {
            "role": "user",
            "content": (
                f"Write a one-paragraph feasibility report for a homeowner in {location} "
                f"who is considering installing a rooftop solar system.\n\n"
                f"Financial details:\n"
                f"- Total system cost: ₹{system_cost:,.0f}\n"
                f"- Government subsidy ({subsidy_percent}%): ₹{subsidy_amount:,.0f}\n"
                f"- Net cost after subsidy: ₹{financials['net_cost_after_subsidy']:,.0f}\n"
                f"- Estimated monthly savings: ₹{monthly_savings:,.0f}\n"
                f"- Payback period: {financials['payback_period_years']} years\n"
                f"- Internal Rate of Return (IRR): {financials['irr_percent']}%\n\n"
                f"Relevant subsidy and eligibility information:\n{policy_answer}\n\n"
                f"Write the report now."
            )
        }
    ]

    # Step 4: Generate
    prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt")
    output = model.generate(**inputs, max_new_tokens=max_new_tokens)

    # Step 5: Extract only the assistant's response
    full_text = tokenizer.decode(output[0], skip_special_tokens=True)
    if "assistant" in full_text.lower():
        report = full_text.split("assistant")[-1].strip()
    else:
        report = full_text.strip()

    return report

In [22]:
# ── Guard: verify all dependencies are live before running the report ──
import sys

missing = []
if 'tokenizer' not in dir():             missing.append("Cell 4  — model + tokenizer")
if 'calculate_solar_feasibility' not in dir(): missing.append("Cell 7  — calculate_solar_feasibility")
if 'index' not in dir():                 missing.append("Cell 14 — embedder + FAISS index")
if 'retrieve_relevant_chunks' not in dir(): missing.append("Cell 15 — retrieve_relevant_chunks")
if 'answer_question' not in dir():       missing.append("Cell 17 — answer_question")

if missing:
    print("❌ Run these cells first, then re-run this cell:\n")
    for m in missing: print(" →", m)
    sys.exit("Missing dependencies.")
else:
    print("✅ All dependencies ready. Proceed to the next cell.")

✅ All dependencies ready. Proceed to the next cell.


## Step 6: Automated Feasibility Report Generation (Agentic Workflow)

This function demonstrates an agentic pipeline: it autonomously chains together the 
financial calculator, the RAG-based policy retrieval, and Granite's generation capability — 
without manual intervention between steps — to produce a complete, plain-language 
feasibility report for a homeowner.

In [24]:
import textwrap, re

report = generate_feasibility_report(
    system_cost=300000,
    subsidy_percent=30,
    monthly_savings=2500,
    location="Gujarat"
)

report = re.sub(r'\.([A-Z])', r'. \1', report)   # fix missing spaces after sentences
print(textwrap.fill(report, width=80))

A rooftop solar system for a homeowner in Gujarat can be a great investment, but
it's important to understand the financial aspects. The total system cost is
₹300,000, and with a 30% government subsidy, the net cost after the subsidy is
₹210,000. This leaves a net savings of ₹2,500 per month. The payback period is
7.0 years, which means the investment will pay off in about seven years. The
Internal Rate of Return (IRR) is 13.71%, which is a good sign that the
investment is promising.


## Responsible AI Considerations

- **Transparency**: All policy information is grounded in retrieved source documents, not 
  the model's own unverified claims.
- **Fairness**: The tool provides equal-quality analysis regardless of system size or 
  location, using the same underlying process for every query.
- **Ethics**: This tool is a decision-support aid, not financial advice — output should be 
  verified against official sources before making investment decisions.
- **Privacy**: No personal or sensitive user data is collected or stored by this prototype.

In [25]:
import os

# Make sure the folder exists
os.makedirs("outputs/sample_reports", exist_ok=True)

# Define a few different test scenarios
scenarios = [
    {"name": "small_home_gujarat", "system_cost": 150000, "subsidy_percent": 40, "monthly_savings": 1500, "location": "Gujarat"},
    {"name": "medium_home_gujarat", "system_cost": 300000, "subsidy_percent": 30, "monthly_savings": 2500, "location": "Gujarat"},
    {"name": "large_msme_national", "system_cost": 800000, "subsidy_percent": 20, "monthly_savings": 6000, "location": "India"},
]

# Generate and save a report for each scenario
for s in scenarios:
    report = generate_feasibility_report(
        system_cost=s["system_cost"],
        subsidy_percent=s["subsidy_percent"],
        monthly_savings=s["monthly_savings"],
        location=s["location"]
    )
    
    filepath = f"outputs/sample_reports/{s['name']}.txt"
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(f"Scenario: {s['name']}\n")
        f.write(f"System cost: ₹{s['system_cost']:,}\n")
        f.write(f"Subsidy: {s['subsidy_percent']}%\n")
        f.write(f"Monthly savings: ₹{s['monthly_savings']:,}\n")
        f.write(f"Location: {s['location']}\n\n")
        f.write("Generated Report:\n")
        f.write(report)
    
    print(f"Saved: {filepath}")

Saved: outputs/sample_reports/small_home_gujarat.txt
Saved: outputs/sample_reports/medium_home_gujarat.txt
Saved: outputs/sample_reports/large_msme_national.txt
